# 1. Өгөгдөл цуглуулах

Эх сурвалж, татсан арга, хэмжээ (бие даалтын Хэсэг 1). Дахин татах: `python -m src.scrape` (кэштэй бол `--offline`), `python -m src.wiki`.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
P, R, F = ROOT / "data" / "processed", ROOT / "reports", ROOT / "figures"
def show(name):
    display(Image(filename=str(F / f"{name}.png"), width=760))

In [2]:
display(Markdown((ROOT / "data" / "raw" / "SNAPSHOT.md").read_text(encoding="utf-8")))

# Өгөгдлийн snapshot

- Татсан огноо: 2026-11-12T23:58
- Эх сурвалж: https://mongolia.basketball (MBA статистикийн систем)
- 2026-2027 улирал энэ огнооны байдлаар дуусаагүй (явцын).

| Улирал | Season ID | Сарууд |
|---|---|---|
| 2025-2026 | c1a0488c-31a5-429d-8254-3b27aa45e702 | 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04 |
| 2026-2027 | ac48cf7e-0249-4450-a2ee-42fc7b97dc16 | 2026-10, 2026-11 |

| Файл | Мөр |
|---|---|
| missing_games | 0 |
| missing_profiles | 0 |
| schedule | 255 |
| games_raw | 252 |
| team_games_raw | 504 |
| player_games_raw | 7130 |
| shots_raw | 33347 |
| pbp_raw | 139595 |
| missing_stats | 0 |
| players_season_raw | 318 |
| players_profile_raw | 260 |


## Эх сурвалжууд

| # | Эх сурвалж | Юу | Хэрхэн авсан |
|---|---|---|---|
| S1 | MBA лигийн албан ёсны статистик (mongolia.basketball) | тоглогчийн улирлын статистик, профайл, тоглолтын хайрцаг оноо | Python (`requests` + `BeautifulSoup`), сайтын дүрмийг (robots.txt) дагаж хүсэлт хооронд ≥1 с |
| S1b | Мөн адил тоглолтын хуудас | шидэлтийн байршил, тоглолтын явцын бичлэг (play-by-play) | хуудсанд нуугдсан JSON-оос; шинэ хүсэлтгүй |
| S2 | Wikipedia — The League (Mongolia), 2023–24 улирал | 2023–24 байр, багийн хот, аваргууд | хуудасны текстээс |
| S3 | Wikipedia/FIBA — шигшээ багийн бүрэлдэхүүн (2026.08) | шигшээ багийн 12 тоглогч | хуудасны текстээс |
| S4 | geoBoundaries | Монголын аймгийн хил | GeoJSON файл |

In [3]:
raw = ROOT / "data" / "raw"
files = sorted(list(raw.glob("*.csv")) + list(raw.glob("*.csv.gz")))
pd.DataFrame([(f.name, len(df := pd.read_csv(f)), len(df.columns)) for f in files], columns=["файл", "мөр", "багана"])

,файл,мөр,багана
0,champions_raw.csv,30,3
1,games_raw.csv,252,14
2,national_team_raw.csv,12,8
3,pbp_raw.csv.gz,139595,14
4,player_games_raw.csv,7130,20
5,players_profile_raw.csv,260,10
6,players_season_raw.csv,318,16
7,schedule.csv,255,12
8,shots_raw.csv,33347,11
9,standings_2023_24_raw.csv,13,7


In [4]:
pd.read_csv(raw / "players_season_raw.csv").head()

,season,rank,player_name,player_slug,team_name,GP,MIN,PPG,RPG,APG,SPG,BPG,PF,FG_pct,TP_pct,EFF
0,2025-2026,1,BYRON Wesley,byron-wesley-2,BUREN SCORE MONGOLIANS,24,32.5,27.9,5.4,3.8,1.9,0.9,1.5,50.3,31.4,25.5
1,2025-2026,2,UORREN Davion,uorren-davion,BCH Knights,26,31.0,26.7,5.9,5.6,3.1,0.7,3.2,49.0,31.5,24.7
2,2025-2026,3,SCRUBB Jay,scrubb-jay,SELENGE BODONS,6,32.5,25.2,7.2,3.5,1.8,2.0,2.8,44.3,36.5,23.8
3,2025-2026,4,MCKENZIE Moore,mckenzie-moore,ZAVKHAN BROTHERS,26,35.9,24.3,9.7,9.0,2.3,1.4,2.7,41.8,30.4,28.3
4,2025-2026,5,BRANDIS Raley Ross,brandis-raley-ross,Khan Falcons,26,30.2,22.3,3.9,4.0,1.2,0.2,2.1,50.7,45.1,20.8


In [5]:
pd.read_csv(raw / "player_games_raw.csv").head()

,game_id,season,team_name,side,name,player_slug,number,pos,dnp,MIN,PTS,REB,AST,STL,BLK,FG,TP,FT,TOV,PF
0,223,2025-2026,XAC BRONCOS,home,ALTANGEREL Azbayar,altangerel-azbayar,7,C,False,33.0,19.0,12.0,0.0,0.0,3.0,7-9,0-0,5-6,1.0,3.0
1,223,2025-2026,XAC BRONCOS,home,MILLER Ian,miller-ian,44,SG,False,33.0,13.0,3.0,5.0,2.0,0.0,4-18,3-11,2-4,3.0,2.0
2,223,2025-2026,XAC BRONCOS,home,BAATAR-ERDENE Barsbold,baatar-erdene-barsbold,5,PG,False,26.0,13.0,6.0,3.0,3.0,1.0,5-13,1-8,2-2,7.0,2.0
3,223,2025-2026,XAC BRONCOS,home,CHULUUNBAYAR Ikhbayar,chuluunbayar-ikhbayar,0,SF,False,22.0,6.0,4.0,0.0,1.0,0.0,3-5,0-2,0-0,1.0,0.0
4,223,2025-2026,XAC BRONCOS,home,SANDAGDORJ Binderyaa,sandagdorj-binderyaa,25,PF,False,21.0,3.0,9.0,1.0,0.0,0.0,1-2,0-0,1-2,0.0,4.0


## Шидэлт ба тоглолтын явц

Тоглолтын хуудас бүр шидэлт бүрийн талбай дээрх байршил (x, y пиксел), оносон эсэх, 2/3 оноо, мөн тоглолтын бүх үйл явдлыг (солилт, дамжуулалт, алдаа ...) цаг, онооны хамт агуулдаг.

In [6]:
display(pd.read_csv(raw / "shots_raw.csv").head(3)); pd.read_csv(raw / "pbp_raw.csv.gz").head(5)

,game_id,seq,side,shot_player_id,player,number,period,made,three,x,y
0,223,19,away,153,TUMENBAYAR Uuganbayar,4,1,True,False,95,48
1,223,22,home,229,CHINGES Temuulen,9,1,False,True,27,22
2,223,24,away,343,ERDENETSETSEG Margad,13,1,False,False,92,54


,game_id,seq,period,clock,side,player_slug,player,number,type,made,points,home,away,text
0,223,1,1,10:00,NaN,NaN,NaN,NaN,start,NaN,0,0,0,Start of Q1
1,223,2,1,10:00,home,chuluunbayar-ikhbayar,CHULUUNBAYAR Ikhbayar,0.0,sub,NaN,0,0,0,substitution in
2,223,3,1,10:00,home,baatar-erdene-barsbold,BAATAR-ERDENE Barsbold,5.0,sub,NaN,0,0,0,substitution in
3,223,4,1,10:00,home,altangerel-azbayar,ALTANGEREL Azbayar,7.0,sub,NaN,0,0,0,substitution in
4,223,5,1,10:00,home,chinges-temuulen-2,CHINGES Temuulen,9.0,sub,NaN,0,0,0,substitution in
